# StreamGraph: first dataset and baseline exploration

Run the terminal setup first. In VS Code select the project `.venv` kernel. If requested, install the optional kernel with `python -m pip install ipykernel`. This notebook explores data/validation outputs; it does not score the final test period.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Open the notebook inside the streamgraph project"
sys.path.insert(0, str(root / "src"))
print("Project:", root)


## 1. Dataset and chronology

Unknown labels are unlabelled, not legitimate transactions. Only known source labels train the baseline models.


In [ ]:
audit_path = root / "artifacts/dataset_audit.json"
assert audit_path.exists(), "Run streamgraph run first"
audit = json.loads(audit_path.read_text())
print("Nodes:", audit["nodes"], "Directed edges:", audit["edges"])
print("Labels:", audit["labels"])
split_table = pd.DataFrame(audit["splits"]).T
split_table[["time_steps", "nodes", "known_labels", "illicit", "unknown"]]


In [ ]:
timeline = pd.DataFrame(audit["per_time_step"])
ax = timeline.plot(x="time_step", y=["licit", "illicit"], figsize=(10, 4), title="Known labels by original time step")
ax.set_xlabel("Original time step")
ax.set_ylabel("Nodes with known labels")
plt.show()


## 2. Measured validation baselines

Epochs and thresholds were selected on validation. These scores are development measurements from one initial seed.


In [ ]:
runs = list((root / "artifacts").glob("baseline_*/results.json"))
assert runs, "Run streamgraph train first"
results_path = max(runs, key=lambda p: p.stat().st_mtime)
results = json.loads(results_path.read_text())
assert results["final_test_evaluated"] is False
print("Run:", results_path.parent.name)
print("Source fitting audit:", results["fit_audit"])
pd.DataFrame(results["models"]).T[["average_precision", "precision_illicit", "recall_illicit", "f1_illicit", "threshold"]]


## 3. A graph snapshot for the next milestone

`x` contains node features; `edge_index` stores local directed endpoints. `y=-1` means unknown. The graph contains only original step 1.


In [ ]:
snapshot_path = root / "data/processed/snapshot_01.npz"
assert snapshot_path.exists(), "Run streamgraph snapshot --step 1 first"
with np.load(snapshot_path, allow_pickle=False) as graph:
    print("x:", graph["x"].shape)
    print("edge_index:", graph["edge_index"].shape)
    print("Known labels:", int(graph["known_mask"].sum()))
    print("Step:", int(graph["time_step"]))
    print("First transaction IDs:", graph["transaction_ids"][:5])


## Next implementation

Train GraphSAGE with the same feature view and source/validation labels. Compare it with the MLP before adding DANN/GRL. Preserve the final-test separation. See `docs/NEXT_MILESTONE.md`.
